## 03 — Multi-day more-metrics (≥ 1 foot standing water)

This notebook computes confusion-matrix metrics for the **image-level** task:

- **Positive class**: the image shows **more than a foot of standing water** (question `q1`).
- **Prediction**: derived by factorizing raw VLM output text (`response_1`) into **yes/no**.
- **Ground truth**: a day-specific annotated set.

Key refactor: for Sep 29 (and other days), we **load chunked raw model outputs** from `notebooks/cambrian/*_{0..5}.csv` and factorize them, instead of relying on pre-aggregated/derived artifacts.

Days included:

- Sep 29 (NYC)
- California
- Jan 10
- Dec 18

Metrics reported:

- Positive predictive value (**PPV / precision**)
- **False omission rate (FOR)**
- **Recall**
- **F1-score**
- **Critical success index (CSI / threat score)**


In [19]:
from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd

# Reproducibility
RANDOM_SEED = 777
rng = np.random.default_rng(RANDOM_SEED)

BASE_DIR = Path("../../")
CAMBRIAN_DIR = BASE_DIR / "notebooks" / "cambrian"

# -------------------------
# Inputs / configuration
# -------------------------

# Sep29 ground-truth labels (contains `image` + numeric `gt`)
SEP29_ANNOTATED_PATH = BASE_DIR / "data" / "processed" / "inspection_set.csv"

# Other-day ground-truth labels (Label Studio exports; contains `image` + string `choice`)
ANNOTATED_SAMPLE_PATHS = {
    "california": CAMBRIAN_DIR / "california_sample_annotated.csv",
    "jan10": CAMBRIAN_DIR / "jan10_sample_annotated.csv",
    "dec18": CAMBRIAN_DIR / "dec18_sample_annotated.csv",
}

# Chunked raw model outputs (free-text `response_1` to factorize)
# Sep29 has two directories; set this to "default" or "13b" depending on which chunks you want.
SEP29_VARIANT = "13b"  # {"default", "13b"}

# Population counts (Total N and Predicted Positive counts) for adjustment.
# These were calculated by scanning the entire dataset for each day.
POPULATION_STATS = {
    "sep29": {"N": 926212, "pos": 1465},     # from notebooks/cambrian/entire_sep29_all.csv
    "california": {"N": 24006, "pos": 7},
    "jan10": {"N": 160206, "pos": 16},
    "dec18": {"N": 198606, "pos": 94},
}


def sep29_chunk_paths() -> list[Path]:
    base = CAMBRIAN_DIR if SEP29_VARIANT == "default" else (CAMBRIAN_DIR / "13b")
    return sorted(base.glob("entire_sep29_[0-9].csv"))


CHUNK_PATHS = {
    "california": sorted(CAMBRIAN_DIR.glob("california_[0-9].csv")),
    "jan10": sorted(CAMBRIAN_DIR.glob("jan10_[0-9].csv")),
    "dec18": sorted(CAMBRIAN_DIR.glob("dec18_[0-9].csv")),
}


def safe_div(num: float, den: float) -> float:
    return float(num) / float(den) if den != 0 else float("nan")


CAMBRIAN_DIR


PosixPath('/share/ju/matt/bayflood/notebooks/cambrian')

In [20]:
import constants as c
from pathlib import Path
PAPER_PATH = Path(c.PAPER_PATH)

In [21]:
# Helpers: parse image IDs, factorize yes/no from response_1, and load chunk predictions

YES_RE = re.compile(r"^\s*[\"']?\s*yes\b", flags=re.IGNORECASE)
NO_RE = re.compile(r"^\s*[\"']?\s*no\b", flags=re.IGNORECASE)


def image_name_from_any(path_or_filename: str) -> str:
    """Normalize various path formats down to the basename filename."""
    s = str(path_or_filename)
    if "?d=" in s:
        s = s.split("?d=", 1)[1]
    return Path(s).name


def factorize_yes_no(text: str) -> int | pd._libs.missing.NAType:
    """Return 1 for Yes, 0 for No, NA if unknown/ambiguous."""
    if pd.isna(text):
        return pd.NA
    s = str(text).strip()
    if YES_RE.match(s):
        return 1
    if NO_RE.match(s):
        return 0
    return pd.NA


def load_chunk_preds(paths: list[Path], wanted_image_names: set[str], chunksize: int = 200_000) -> pd.DataFrame:
    """Stream CSV chunks and keep predictions only for the labeled images we care about."""
    if len(paths) == 0:
        raise FileNotFoundError("No chunk files found.")

    kept = []
    for p in paths:
        for chunk in pd.read_csv(p, chunksize=chunksize):
            # Identify image path column
            if "image_path" in chunk.columns:
                col = "image_path"
            elif "img_path" in chunk.columns:
                col = "img_path"
                raise KeyError(f"{p} missing an image path column (expected image_path or img_path)")

            if "response_1" not in chunk.columns:
                raise KeyError(f"{p} missing response_1")

            sub = chunk[[col, "response_1"]].copy()
            sub["image_name"] = sub[col].map(image_name_from_any)
            sub = sub[sub["image_name"].isin(wanted_image_names)]
            if len(sub) == 0:
                continue

            sub["pred_bin"] = sub["response_1"].map(factorize_yes_no)
            kept.append(sub[["image_name", "pred_bin"]])

    if len(kept) == 0:
        return pd.DataFrame({"image_name": [], "pred_bin": []})

    out = pd.concat(kept, ignore_index=True)
    # De-dup just in case (keep first observed)
    out = out.drop_duplicates(subset=["image_name"], keep="first")
    return out


# quick smoke checks
sep29_chunk_paths()[:2], {k: len(v) for k, v in CHUNK_PATHS.items()}


([PosixPath('/share/ju/matt/bayflood/notebooks/cambrian/13b/entire_sep29_0.csv'),
  PosixPath('/share/ju/matt/bayflood/notebooks/cambrian/13b/entire_sep29_1.csv')],
 {'california': 6, 'jan10': 6, 'dec18': 6})

In [22]:
# Load each day's ground truth, join against chunked predictions, and compute metrics

def confusion_and_metrics_weighted(tp, fp, tn, fn, w1, w0) -> dict[str, float]:
    """Compute population-adjusted metrics using stratified sampling weights."""
    tp_pop = tp * w1
    fp_pop = fp * w1
    tn_pop = tn * w0
    fn_pop = fn * w0
    n_pop = tp_pop + fp_pop + tn_pop + fn_pop

    return {
        "n_eval_sample": int(tp + fp + tn + fn),
        "n_pos_sample": int(tp + fn),
        "n_eval": int(n_pop),
        "n_pos": int(tp_pop + fn_pop),
        "n_pred_pos": int(tp_pop + fp_pop),
        "n_pred_neg": int(tn_pop + fn_pop),
        "prevalence_gt": safe_div(tp_pop + fn_pop, n_pop),
        "predicted_positive_rate": safe_div(tp_pop + fp_pop, n_pop),
        "ppv_precision": safe_div(tp_pop, tp_pop + fp_pop),
        "false_omission_rate": safe_div(fn_pop, fn_pop + tn_pop),
        "recall": safe_div(tp_pop, tp_pop + fn_pop),
        "f1": safe_div(2 * tp_pop, 2 * tp_pop + fp_pop + fn_pop),
        "csi_threat_score": safe_div(tp_pop, tp_pop + fp_pop + fn_pop),
    }


def eval_day_weighted(day, gt_df, paths):
    wanted = set(gt_df["image_name"].unique())
    preds = load_chunk_preds(paths, wanted)
    merged = gt_df.merge(preds, on="image_name", how="left")
    
    n_gt = len(gt_df)
    n_found = int(merged["pred_bin"].notna().sum())
    diagnostics.append({"day": day, "n_gt": n_gt, "n_pred_found": n_found, "coverage": n_found / n_gt if n_gt else float("nan")})

    work = merged.dropna(subset=["pred_bin"]).copy()
    work["pred_bin"] = work["pred_bin"].astype(int)
    
    tp = int(((work["gt_bin"] == 1) & (work["pred_bin"] == 1)).sum())
    fp = int(((work["gt_bin"] == 0) & (work["pred_bin"] == 1)).sum())
    tn = int(((work["gt_bin"] == 0) & (work["pred_bin"] == 0)).sum())
    fn = int(((work["gt_bin"] == 1) & (work["pred_bin"] == 0)).sum())

    pop_n = POPULATION_STATS[day]["N"]
    pop_pos = POPULATION_STATS[day]["pos"]
    pop_neg = pop_n - pop_pos

    n_pos_pred = tp + fp
    n_neg_pred = tn + fn

    w1 = pop_pos / n_pos_pred if n_pos_pred > 0 else 0
    w0 = pop_neg / n_neg_pred if n_neg_pred > 0 else 0
    
    res = {"day": day, "variant": SEP29_VARIANT if day == "sep29" else "default"}
    res |= confusion_and_metrics_weighted(tp, fp, tn, fn, w1, w0)
    return res, merged


def load_sep29_gt() -> pd.DataFrame:
    df = pd.read_csv(SEP29_ANNOTATED_PATH)
    if "image" not in df.columns or "response_1" not in df.columns:
        raise KeyError("Expected columns `image` and `gt` in inspection_set.csv")
    out = df[["image", "choice"]].copy()
    out["image_name"] = out["image"].map(image_name_from_any)

    # gt column = 1 if 'Yes' in response_1, 0 otherwise
    out["gt_bin"] = (out["choice"].astype(str) == "Flooded road").astype(int)

    out = out.dropna(subset=["gt_bin"]).copy()
    out["gt_bin"] = out["gt_bin"].astype(int)
    return out[["image_name", "gt_bin"]]


def load_otherday_gt(day: str, path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "image" not in df.columns or "choice" not in df.columns:
        raise KeyError(f"Expected columns `image` and `choice` in {path}")
    out = df[["image", "choice"]].copy()
    out["image_name"] = out["image"].map(image_name_from_any)
    out["gt_bin"] = (out["choice"].astype(str) == "Flooded road").astype(int)
    return out[["image_name", "gt_bin"]]


results = []
diagnostics = []

# Sep29
sep29_gt = load_sep29_gt()
res, sep29_merged = eval_day_weighted("sep29", sep29_gt, sep29_chunk_paths())
results.append(res)

# Other days
for day, gt_path in ANNOTATED_SAMPLE_PATHS.items():
    gt = load_otherday_gt(day, gt_path)
    res, _ = eval_day_weighted(day, gt, CHUNK_PATHS[day])
    results.append(res)

pd.DataFrame(results).set_index("day").sort_index()


,variant,n_eval_sample,n_pos_sample,n_eval,n_pos,n_pred_pos,n_pred_neg,prevalence_gt,predicted_positive_rate,ppv_precision,false_omission_rate,recall,f1,csi_threat_score
day,,,,,,,,,,,,,,
california,default,257,1,24006,1,7,23999,0.000042,0.000292,0.142857,0.000,1.00000,0.250000,0.142857
dec18,default,344,66,198606,66,94,198512,0.000332,0.000473,0.702128,0.000,1.00000,0.825000,0.702128
jan10,default,266,13,160206,13,16,160190,0.000081,0.000100,0.812500,0.000,1.00000,0.896552,0.812500
sep29,13b,1000,332,926211,6512,1465,924746,0.007031,0.001582,0.658000,0.006,0.14802,0.241674,0.137445


In [23]:
# Diagnostics: how many labeled images were found in the chunk predictions?

pd.DataFrame(diagnostics).set_index("day").sort_index()


,n_gt,n_pred_found,coverage
day,,,
california,257,257,1.0
dec18,344,344,1.0
jan10,266,266,1.0
sep29,1000,1000,1.0


In [24]:
# Optional: bootstrap SD (per day) over population resampling

# pd max columns display to 50 
pd.set_option('display.max_columns', 50)

DO_BOOTSTRAP = True
N_BOOT = 100000


def bootstrap_stats(y_true: np.ndarray, y_pred: np.ndarray, pop_n: int, pop_pos: int, n_boot: int) -> dict[str, float]:
    pos_idx = np.where(y_pred == 1)[0]
    neg_idx = np.where(y_pred == 0)[0]
    
    n1_sample = len(pos_idx)
    n0_sample = len(neg_idx)
    
    # Observed stratum success rates (P(y=1 | y_hat))
    p1_obs = (y_true[pos_idx] == 1).mean() if n1_sample > 0 else 0
    p0_obs = (y_true[neg_idx] == 1).mean() if n0_sample > 0 else 0
    
    # Population counts (scanned from total dataset)
    n1_pop = pop_pos
    n0_pop = pop_n - n1_pop
    
    keys = ["ppv_precision", "false_omission_rate", "recall", "f1", "csi_threat_score", "prevalence_gt", "predicted_positive_rate", "n_pred_pos", "n_pred_neg"]
    boot = {k: [] for k in keys}

    for _ in range(n_boot):
        # Sample conditional probabilities from the annotation binomials
        # (Equivalent to resampling the ground truth pool with replacement)
        p1 = rng.binomial(n1_sample, p1_obs) / n1_sample if n1_sample > 0 else 0
        p0 = rng.binomial(n0_sample, p0_obs) / n0_sample if n0_sample > 0 else 0
        
        # Scale sampled probabilities to population counts
        tp = n1_pop * p1
        fp = n1_pop * (1 - p1)
        fn = n0_pop * p0
        tn = n0_pop * (1 - p0)
        
        # Compute metrics on these population-scale counts
        m = confusion_and_metrics_weighted(tp, fp, tn, fn, 1.0, 1.0)
        for k in keys:
            boot[k].append(m[k])

    out = {}
    for k in keys:
        arr = np.asarray(boot[k], dtype=float)
        out[f"{k}_sd"] = float(np.nanstd(arr))
        out[f"{k}_ci_lower"] = float(np.nanquantile(arr, 0.025))
        out[f"{k}_ci_upper"] = float(np.nanquantile(arr, 0.975))
    return out


if DO_BOOTSTRAP:
    rows = []
    for day in ["sep29", "california", "jan10", "dec18"]:
        if day == "sep29":
            gt = load_sep29_gt()
            paths = sep29_chunk_paths()
        else:
            gt = load_otherday_gt(day, ANNOTATED_SAMPLE_PATHS[day])
            paths = CHUNK_PATHS[day]
            
        wanted = set(gt["image_name"].unique())
        preds = load_chunk_preds(paths, wanted)
        merged = gt.merge(preds, on="image_name", how="left").dropna(subset=["pred_bin"]).copy()
        merged["pred_bin"] = merged["pred_bin"].astype(int)
        
        y_true = merged["gt_bin"].to_numpy(dtype=int)
        y_pred = merged["pred_bin"].to_numpy(dtype=int)
        
        pop_stats = POPULATION_STATS[day]
        
        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())
        
        w1 = pop_stats["pos"] / (tp + fp) if (tp + fp) > 0 else 0
        w0 = (pop_stats["N"] - pop_stats["pos"]) / (tn + fn) if (tn + fn) > 0 else 0
        
        base = {"day": day}
        base |= confusion_and_metrics_weighted(tp, fp, tn, fn, w1, w0)
        base |= bootstrap_stats(y_true, y_pred, pop_stats["N"], pop_stats["pos"], N_BOOT)
        rows.append(base)

    display(pd.DataFrame(rows).set_index("day").sort_index())
else:
    print("Set DO_BOOTSTRAP=True to compute per-day bootstrap confidence intervals.")


,n_eval_sample,n_pos_sample,n_eval,n_pos,n_pred_pos,n_pred_neg,prevalence_gt,predicted_positive_rate,ppv_precision,false_omission_rate,recall,f1,csi_threat_score,ppv_precision_sd,ppv_precision_ci_lower,ppv_precision_ci_upper,false_omission_rate_sd,false_omission_rate_ci_lower,false_omission_rate_ci_upper,recall_sd,recall_ci_lower,recall_ci_upper,f1_sd,f1_ci_lower,f1_ci_upper,csi_threat_score_sd,csi_threat_score_ci_lower,csi_threat_score_ci_upper,prevalence_gt_sd,prevalence_gt_ci_lower,prevalence_gt_ci_upper,predicted_positive_rate_sd,predicted_positive_rate_ci_lower,predicted_positive_rate_ci_upper,n_pred_pos_sd,n_pred_pos_ci_lower,n_pred_pos_ci_upper,n_pred_neg_sd,n_pred_neg_ci_lower,n_pred_neg_ci_upper
day,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
california,257,1,24006,1,7,23999,0.000042,0.000292,0.142857,0.000,1.00000,0.250000,0.142857,0.132029,0.000000,0.428571,0.000000,0.0,0.000,0.000000,1.000000,1.0,0.193151,0.000000,0.600000,0.132029,0.000000,0.428571,0.000038,0.000000,0.000125,5.421011e-20,0.000292,0.000292,0.0,7.0,7.0,0.000000,23999.0,23999.0
dec18,344,66,198606,66,94,198512,0.000332,0.000473,0.702128,0.000,1.00000,0.825000,0.702128,0.047112,0.606383,0.787234,0.000000,0.0,0.000,0.000000,1.000000,1.0,0.032714,0.754967,0.880952,0.047112,0.606383,0.787234,0.000022,0.000287,0.000373,0.000000e+00,0.000473,0.000473,0.0,94.0,94.0,0.000000,198512.0,198512.0
jan10,266,13,160206,13,16,160190,0.000081,0.000100,0.812500,0.000,1.00000,0.896552,0.812500,0.097690,0.625000,1.000000,0.000000,0.0,0.000,0.000000,1.000000,1.0,0.061445,0.769231,1.000000,0.097690,0.625000,1.000000,0.000010,0.000062,0.000100,0.000000e+00,0.000100,0.000100,0.0,16.0,16.0,0.000000,160190.0,160190.0
sep29,1000,332,926211,6512,1465,924746,0.007031,0.001582,0.658000,0.006,0.14802,0.241674,0.137445,0.021208,0.616000,0.700000,0.003448,0.0,0.014,0.196707,0.069886,1.0,0.148913,0.126462,0.793727,0.127411,0.067499,0.658000,0.003443,0.001041,0.015006,4.335296e-19,0.001582,0.001582,0.0,1465.0,1465.0,0.030482,924747.0,924747.0


In [25]:
# Simplified LaTeX Table Generation

DAY_LABELS = {
    "sep29": "9/29/23, NYC",
    "dec18": "12/18/23, NYC",
    "jan10": "1/10/24, NYC",
    "california": "2/10/24, SF Bay"
}
DAY_ORDER = ["sep29", "dec18", "jan10", "california"]

# 1. Prepare data
data_source = rows if (DO_BOOTSTRAP and 'rows' in locals()) else results
df_base = pd.DataFrame(data_source)
df_base['day'] = pd.Categorical(df_base['day'], categories=DAY_ORDER, ordered=True)
df_base = df_base.sort_values('day')

# 2. Define simplified metrics
metrics_simplified = {
    "ppv_precision": "$P(y=1|\\hat{y}=1)$",
    "false_omission_rate": "$P(y=1|\\hat{y}=0)$"
}

def format_val_simple(row, col):
    val = row[col]
    if pd.isna(val): return "--"
    
    def to_sci_compact(v, low=None, high=None):
        def _fmt(val):
            s = f"{val:.2e}"
            b, e = s.split("e")
            return f"{b} \\cdot 10^{{{int(e)}}}"
        if low is None or pd.isna(low) or high is None or pd.isna(high):
            return _fmt(v)
        else:
            return f"{_fmt(v)} \\ [ {_fmt(low)}, {_fmt(high)} ]"

    # Use scientific notation for low-prevalence metrics (FOR)
    if col in ["false_omission_rate"]:
        low = row.get(f"{col}_ci_lower")
        high = row.get(f"{col}_ci_upper")
        return f"${to_sci_compact(val, low, high)}$"
    
    # Standard decimal for other metrics
    prec = 3
    low = row.get(f"{col}_ci_lower")
    high = row.get(f"{col}_ci_upper")
    if not pd.isna(low) and not pd.isna(high):
        return f"${val:.{prec}f} \\ [{low:.{prec}f}, {high:.{prec}f}]$"
    return f"${val:.{prec}f}$"

# 3. Build simplified table
table_rows = []
for m_key, m_label in metrics_simplified.items():
    row_vals = [m_label]
    for day in DAY_ORDER:
        day_row = df_base[df_base['day'] == day].iloc[0]
        row_vals.append(format_val_simple(day_row, m_key))
    table_rows.append(row_vals)

df_final_simple = pd.DataFrame(table_rows, columns=["Metric"] + [DAY_LABELS[d] for d in DAY_ORDER]).set_index("Metric").T
df_final_simple.index.name = None

# 4. Generate LaTeX
caption = "Simplified performance metrics across all experiment days."
latex_code_simple = df_final_simple.to_latex(
    index=True,
    caption=caption,
    label="tab:vlm_simplified",
    escape=False,
    column_format="l" + "p{4.5cm}" * len(df_final_simple.columns),
    bold_rows=False
)
latex_code_simple = latex_code_simple.replace("\\begin{table}", "\\begin{table}\n\\small")

print("% Simplified LaTeX Table Output")
print(latex_code_simple)

with open(PAPER_PATH / "tables" / "vlm_simplified_performance_all_days.tex", "w") as f:
    f.write(latex_code_simple)


% Simplified LaTeX Table Output
\begin{table}
\small
\caption{Simplified performance metrics across all experiment days.}
\label{tab:vlm_simplified}
\begin{tabular}{lp{4.5cm}p{4.5cm}}
\toprule
Metric & $P(y=1|\hat{y}=1)$ & $P(y=1|\hat{y}=0)$ \\
\midrule
9/29/23, NYC & $0.658 \ [0.616, 0.700]$ & $6.00 \cdot 10^{-3} \ [ 0.00 \cdot 10^{0}, 1.40 \cdot 10^{-2} ]$ \\
12/18/23, NYC & $0.702 \ [0.606, 0.787]$ & $0.00 \cdot 10^{0} \ [ 0.00 \cdot 10^{0}, 0.00 \cdot 10^{0} ]$ \\
1/10/24, NYC & $0.812 \ [0.625, 1.000]$ & $0.00 \cdot 10^{0} \ [ 0.00 \cdot 10^{0}, 0.00 \cdot 10^{0} ]$ \\
2/10/24, SF Bay & $0.143 \ [0.000, 0.429]$ & $0.00 \cdot 10^{0} \ [ 0.00 \cdot 10^{0}, 0.00 \cdot 10^{0} ]$ \\
\bottomrule
\end{tabular}
\end{table}



In [26]:
# Alternate Version: Raw Sample Metrics with Population-Adjusted Prevalence and Scientific Notation
# This follows the logic in bootstrap_matt_recall_f1.ipynb:
# p(y=1) = p(yhat=1)*p(y=1|yhat=1) + p(yhat=0)*p(y=1|yhat=0)

def raw_metrics_bootstrap_pop(tp, fp, tn, fn, p_yhat_1, n_boot=10000) -> dict[str, dict]:
    """Compute metrics with population-adjusted prevalence and bootstrap CIs."""
    n_pos_pred = tp + fp
    n_neg_pred = tn + fn
    
    # Observed rates (stratum-specific)
    p_ppv_obs = tp / n_pos_pred if n_pos_pred > 0 else 0
    p_for_obs = fn / n_neg_pred if n_neg_pred > 0 else 0
    
    # Observed population prevalence
    p_prev_obs = p_yhat_1 * p_ppv_obs + (1 - p_yhat_1) * p_for_obs
    
    # Bootstrap arrays
    boot_prev = []
    boot_ppv = []
    boot_for = []
    
    for _ in range(n_boot):
        # Sample stratum-specific rates from Binomial distributions
        s_ppv = np.random.binomial(n_pos_pred, p_ppv_obs) / n_pos_pred if n_pos_pred > 0 else 0
        s_for = np.random.binomial(n_neg_pred, p_for_obs) / n_neg_pred if n_neg_pred > 0 else 0
        
        # Compute population prevalence for this bootstrap sample
        s_prev = p_yhat_1 * s_ppv + (1 - p_yhat_1) * s_for
        
        boot_prev.append(s_prev)
        boot_ppv.append(s_ppv)
        boot_for.append(s_for)
        
    def get_stats(arr, val_obs):
        return {
            "val": val_obs,
            "low": np.percentile(arr, 2.5),
            "high": np.percentile(arr, 97.5)
        }

    return {
        "prevalence": get_stats(boot_prev, p_prev_obs),
        "ppv": get_stats(boot_ppv, p_ppv_obs),
        "for_val": get_stats(boot_for, p_for_obs),
    }

raw_rows = []
for day in ["sep29", "dec18", "jan10", "california"]:
    if day == "sep29":
        gt = load_sep29_gt()
        paths = sep29_chunk_paths()
    else:
        gt = load_otherday_gt(day, ANNOTATED_SAMPLE_PATHS[day])
        paths = CHUNK_PATHS[day]
        
    wanted = set(gt["image_name"].unique())
    preds = load_chunk_preds(paths, wanted)
    merged = gt.merge(preds, on="image_name", how="left").dropna(subset=["pred_bin"]).copy()
    merged["pred_bin"] = merged["pred_bin"].astype(int)
    
    y_true = merged["gt_bin"].to_numpy(dtype=int)
    y_pred = merged["pred_bin"].to_numpy(dtype=int)
    
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    
    # Get population predicted positive rate from POPULATION_STATS
    pop_stats = POPULATION_STATS[day]
    p_yhat_1 = pop_stats["pos"] / pop_stats["N"]
    
    day_label = DAY_LABELS[day]
    date_str, loc_str = day_label.split(", ")
    
    m = raw_metrics_bootstrap_pop(tp, fp, tn, fn, p_yhat_1)
    raw_rows.append({
        "Date": date_str,
        "Location": loc_str,
        "prev": m["prevalence"],
        "ppv": m["ppv"],
        "for_val": m["for_val"]
    })

caption = "Validation of VLM performance across multiple days and locations. Results reported are for our preferred model (Cambrian-1-13B). Classified positives ($\\hat y = 1$) are much likelier to show flooding ($y = 1$) than classified negatives across all four days. Prevalence $p(y=1)$ is estimated using population-adjusted weights. Metrics include 95% bootstrap confidence intervals computed from raw sample counts."
label = "tab:other-days-performance"

def fmt_sci(d):
    if pd.isna(d["val"]): return "--"
    def _fmt(v):
        if v == 0: return "0.00"
        s = "{:.2e}".format(v)
        b, e = s.split("e")
        return "{} \\cdot 10^{{{}}}".format(b, int(e))
    return "${} \\ [ {}, {} ]$".format(_fmt(d["val"]), _fmt(d["low"]), _fmt(d["high"]))

def fmt_dec(d):
    if pd.isna(d["val"]): return "--"
    return "{:.3f} [{:.3f}, {:.3f}]".format(d["val"], d["low"], d["high"])

latex_lines = [
    "\\begin{table}[h!]",
    "\\centering",
    "\\small",
    "\\begin{tabular}{llccc}",
    "\\toprule",
    "Date & Location & $p(y=1)$ & $p(y=1|\\hat y = 1)$ & $p(y=1|\\hat y = 0)$ \\\\",
    "\\midrule"
]

for r in raw_rows:
    row_str = "{} & {} & {} & {} & {} \\\\".format(
        r["Date"], r["Location"], fmt_sci(r["prev"]), fmt_dec(r["ppv"]), fmt_dec(r["for_val"])
    )
    latex_lines.append(row_str)

latex_lines.extend([
    "\\bottomrule",
    "\\end{tabular}",
    "\\caption{" + caption + "}",
    "\\label{" + label + "}",
    "\\end{table}"
])

latex_raw = "\n".join(latex_lines)
print("% Raw Metrics LaTeX Table with Scientific Notation for Prevalence")
print(latex_raw)


% Raw Metrics LaTeX Table with Scientific Notation for Prevalence
\begin{table}[h!]
\centering
\small
\begin{tabular}{llccc}
\toprule
Date & Location & $p(y=1)$ & $p(y=1|\hat y = 1)$ & $p(y=1|\hat y = 0)$ \\
\midrule
9/29/23 & NYC & $7.03 \cdot 10^{-3} \ [ 1.04 \cdot 10^{-3}, 1.50 \cdot 10^{-2} ]$ & 0.658 [0.616, 0.698] & 0.006 [0.000, 0.014] \\
12/18/23 & NYC & $3.32 \cdot 10^{-4} \ [ 2.87 \cdot 10^{-4}, 3.73 \cdot 10^{-4} ]$ & 0.702 [0.606, 0.787] & 0.000 [0.000, 0.000] \\
1/10/24 & NYC & $8.11 \cdot 10^{-5} \ [ 6.24 \cdot 10^{-5}, 9.99 \cdot 10^{-5} ]$ & 0.812 [0.625, 1.000] & 0.000 [0.000, 0.000] \\
2/10/24 & SF Bay & $4.17 \cdot 10^{-5} \ [ 0.00, 1.25 \cdot 10^{-4} ]$ & 0.143 [0.000, 0.429] & 0.000 [0.000, 0.000] \\
\bottomrule
\end{tabular}
\caption{Validation of VLM performance across multiple days and locations. Results reported are for our preferred model (Cambrian-1-13B). Classified positives ($\hat y = 1$) are much likelier to show flooding ($y = 1$) than classified negative